# Xarray engine: chemical and optical dimensions

This notebook demonstrates how earthkit-data deals with parameters which involve:
- a type of aerosol or chemical constituent (for example, CAMS atmospheric composition data),
- radiation wavelength (for example, CAMS aerosol optical depth data).

In [1]:
import earthkit.data as ekd

### Chemical constituent dimension

Parameters which involve aerosol type or chemical constituent type should have a relevant metadata exposed via
earthkit's `parameter.chem` and `parameter.chem_long_name` metadata keys. To illustrate this, consider
the following CAMS atmospheric composition dataset:

In [2]:
fl_chem = ekd.from_source("sample", "chem-cams.grib").to_fieldlist()

In [3]:
fl_chem.ls(keys=["parameter.variable", "parameter.chem", "parameter.chem_long_name"])

,parameter.variable,parameter.chem,parameter.chem_long_name
0,mass_mixrat,CO,Carbon monoxide
1,mass_mixrat,HCHO,Formaldehyde
2,mass_mixrat,O3,Ozone


In [4]:
fl_chem[0]

number_of_values,84
array_type,ndarray
array_dtype,float64
variable,mass_mixrat
standard_name,mass_fraction_of_carbon_monoxide_in_air
long_name,Mass mixing ratio
units,dimensionless
chem,CO
chem_long_name,Carbon monoxide
valid_datetime,2011-09-29 00:00:00
base_datetime,2011-09-29 00:00:00


The conversion into Xarray constructs the ``"chem"`` dimension automatically:

In [5]:
ds_chem = fl_chem.to_xarray()
ds_chem

<xarray.Dataset> Size: 2kB
Dimensions:      (chem: 3, latitude: 7, longitude: 12)
Coordinates:
  * chem         (chem) <U4 48B 'CO' 'HCHO' 'O3'
  * latitude     (latitude) float64 56B 90.0 60.0 30.0 0.0 -30.0 -60.0 -90.0
  * longitude    (longitude) float64 96B 0.0 30.0 60.0 ... 270.0 300.0 330.0
Data variables:
    mass_mixrat  (chem, latitude, longitude) float64 2kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

The information included in ``"parameter.chem_long_name"`` can be easily added as an auxiliary coordinate (depending on the dimension ``"chem"``), using ``aux_coords`` option:

In [6]:
ds_chem2 = fl_chem.to_xarray(
    aux_coords={"chem_long_name": ("parameter.chem_long_name", "chem")},
)
ds_chem2

<xarray.Dataset> Size: 2kB
Dimensions:         (chem: 3, latitude: 7, longitude: 12)
Coordinates:
  * chem            (chem) <U4 48B 'CO' 'HCHO' 'O3'
    chem_long_name  (chem) <U15 180B ...
  * latitude        (latitude) float64 56B 90.0 60.0 30.0 0.0 -30.0 -60.0 -90.0
  * longitude       (longitude) float64 96B 0.0 30.0 60.0 ... 270.0 300.0 330.0
Data variables:
    mass_mixrat     (chem, latitude, longitude) float64 2kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

In [7]:
ds_chem2["chem_long_name"].load()

<xarray.DataArray 'chem_long_name' (chem: 3)> Size: 180B
array(['Carbon monoxide', 'Formaldehyde', 'Ozone'], dtype='<U15')
Coordinates:
  * chem            (chem) <U4 48B 'CO' 'HCHO' 'O3'
    chem_long_name  (chem) <U15 180B 'Carbon monoxide' 'Formaldehyde' 'Ozone'

### Wavelength dimension

Optical parameters may depend on the radiation wavelength at which they are measured, modelled or reported.
For example, the following CAMS dataset provides a analysis for aerosol optical depth at 550 and 800 nm:

In [8]:
fl_opt = ekd.from_source("sample", "optical-cams.grib").to_fieldlist()
fl_opt.ls(
    keys=[
        "parameter.variable",
        "parameter.long_name",
        "parameter.chem",
        "parameter.chem_long_name",
        "parameter.wavelength",
    ]
)

,parameter.variable,parameter.long_name,parameter.chem,parameter.chem_long_name,parameter.wavelength
0,aod,Aerosol optical depth,aer_total,Total aerosol,550
1,aod,Aerosol optical depth,aer_total,Total aerosol,800
2,aod,Aerosol optical depth,aer_sm,Aerosol small mode,550
3,aod,Aerosol optical depth,aer_sm,Aerosol small mode,800


In [9]:
fl_opt[0]

number_of_values,84
array_type,ndarray
array_dtype,float64
variable,aod
standard_name,unknown
long_name,Aerosol optical depth
units,Numeric
chem,aer_total
chem_long_name,Total aerosol
wavelength,550
wavelength_bounds,None


In [10]:
ds_opt = fl_opt.to_xarray()
ds_opt

<xarray.Dataset> Size: 3kB
Dimensions:     (chem: 2, wavelength: 2, latitude: 7, longitude: 12)
Coordinates:
  * chem        (chem) <U9 72B 'aer_sm' 'aer_total'
  * wavelength  (wavelength) int64 16B 550 800
  * latitude    (latitude) float64 56B 90.0 60.0 30.0 0.0 -30.0 -60.0 -90.0
  * longitude   (longitude) float64 96B 0.0 30.0 60.0 90.0 ... 270.0 300.0 330.0
Data variables:
    aod         (chem, wavelength, latitude, longitude) float64 3kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF